# B3 - Person B post-pilot training and handoff

Use this after B2 has produced its pilot CSV and P checkpoints. Attach `bsd-data` with the manifest-matched `BSD400/` images and the latest `denoise-B-results` version containing `artifacts/checkpoints/`, `artifacts/tuning/`, and, for inference, `artifacts/noisy/`. B3 defaults to reviewing the pilot; change its stage only when the previous decision is complete.

## 1. Check Lambda Pilot Results
Review `artifacts/tuning/lambda_pilot_unet.csv` and the three pilot checkpoint logs. Compare mean validation PSNR for lambda 0.1, 0.5, and 1.0 using B's 20 validation images. A's legacy pilot selected 0.5, but its validation images differ from B's, so use B's results and agree with A rather than copying the value automatically.

## 2. Freeze Shared Lambda in Protocol
Agree on one shared lambda with Person A. Set `LAMBDA_FROZEN` in `common/protocol.py`, commit and push that change, then replace `PINNED_COMMIT` in the config cell with the new commit hash. Do not start G training while the value is still `None`.

## 3. Train G Variant Models
Set `RUN_STAGE = "train_G"` to run `train_G_s0` and `train_G_s1` with the shared frozen lambda and 20,000-step budget. The runs resume from `last.pt`; save a Kaggle version between sessions if they pause.

## 4. Run Inference with Trained Models
After P and G checkpoints are complete, set `RUN_STAGE = "infer"` and keep the desired IDs in `MODEL_IDS_TO_INFER`. The default list includes P and G for seeds 0 and 1.

## 5. Review and Validate Outputs
Check each checkpoint folder for `best.pt`, `last.pt`, `meta.json`, and `train_log.csv`. Review validation PSNR curves and confirm inference PNGs and runtime CSVs were written under `artifacts/outputs/` and `artifacts/results/`.

## 6. Publish Results and Update Documentation
Save/publish `artifacts/checkpoints/`, `artifacts/outputs/`, `artifacts/results/`, and `artifacts/logs/` as a new `denoise-B-results` dataset version. Record its version and update `docs/model_provenance.md` from each model's `meta.json`.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# ---- CONFIG ----
REPO_URL = "https://github.com/abhinavpareek655/denoise-study.git"
BRANCH = "main"
PINNED_COMMIT = "faf2da1"  # update to the protocol-freeze commit before G training
RUN_STAGE = "review"       # review | train_G | infer | verify
MODEL_IDS_TO_INFER = ["unet_P_s0", "unet_P_s1", "unet_G_s0", "unet_G_s1"]
SESSION_LIMIT_H = 9.0
MAX_HOURS_PER_RUN = 8.5

WORK = Path("/kaggle/working")
REPO = WORK / "denoise-study"
INPUT = Path("/kaggle/input")
assert Path("/kaggle").exists(), "Run this notebook on Kaggle."


def run(cmd, check=True):
    print("$", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, cwd=REPO, check=check, text=True)


def find_data_root():
    for folder in INPUT.rglob("BSD400"):
        if folder.is_dir():
            return folder.parent
    archives = sorted(INPUT.rglob("bsd400.zip"))
    assert archives, "Attach BSD400/ or bsd400.zip as a Kaggle input."
    target = WORK / "bsd-data"
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as zf:
        root = target.resolve()
        for item in zf.infolist():
            assert (target / item.filename).resolve().is_relative_to(root), f"unsafe ZIP path: {item.filename}"
        zf.extractall(target)
    folders = list(target.rglob("BSD400"))
    assert folders, "The attached ZIP has no BSD400/ directory."
    return folders[0].parent


if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], cwd=WORK, check=True)
subprocess.run(["git", "fetch", "--all", "--prune"], cwd=REPO, check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], cwd=REPO, check=True)
print("code commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())

DATA = find_data_root()
os.environ["DENOISE_DATA"] = str(DATA)
sys.path.insert(0, str(REPO))
from common.io import sha256_file
from common.manifest import load_manifest, select, validate_manifest

manifest = load_manifest(REPO / "artifacts" / "manifest.json")
validate_manifest(manifest)
bsd400 = select(manifest, "BSD400")
assert len(bsd400) == 400, f"Expected 400 BSD400 records, found {len(bsd400)}"
bad = [r["relative_path"] for r in bsd400 if sha256_file(DATA / r["relative_path"]) != r["sha256"]]
assert not bad, f"BSD400 hash mismatch with pinned manifest: {bad[:5]}"
print("BSD400 manifest check passed:", len(bsd400), "images")

# Import any compatible artifacts tree from attached Kaggle result datasets.
def artifact_roots():
    roots = {p for p in INPUT.rglob("artifacts") if p.is_dir()}
    for item in INPUT.rglob("last.pt"):
        if item.parent.parent.name == "checkpoints":
            roots.add(item.parent.parent.parent)
    for item in INPUT.rglob("run_meta.json"):
        if item.parent.parent.name == "outputs":
            roots.add(item.parent.parent.parent)
    for item in INPUT.rglob("lambda_pilot_unet.csv"):
        if item.parent.name == "tuning":
            roots.add(item.parent.parent)
    return sorted(roots)


ARTIFACTS = REPO / "artifacts"
IMPORTED_ROOTS = artifact_roots()
for source in IMPORTED_ROOTS:
    for name in ("checkpoints", "outputs", "results", "logs", "tuning", "noisy"):
        src, dst = source / name, ARTIFACTS / name
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print("imported", src)
print("artifact roots found:", IMPORTED_ROOTS if IMPORTED_ROOTS else "none")


In [ ]:
import time
SESSION_START = time.time()

In [ ]:
import pandas as pd
from common import protocol as PR

PILOT_CSV = ARTIFACTS / "tuning" / "lambda_pilot_unet.csv"
PILOT_LOGS = sorted((ARTIFACTS / "checkpoints").glob("unet_G_pilot_lam*_s*/train_log.csv"))
if PILOT_CSV.is_file():
    pilot = pd.read_csv(PILOT_CSV)
    required = {"lambda", "val_psnr_mean", "val_grad_err_mean"}
    assert required.issubset(pilot.columns), f"Pilot CSV is missing columns: {sorted(required - set(pilot.columns))}"
    print("B U-Net validation-only lambda pilot:")
    display(pilot.sort_values("lambda").reset_index(drop=True))
    best = pilot.loc[pilot["val_psnr_mean"].idxmax()]
    print(f"Highest mean validation PSNR candidate: lambda={best['lambda']}, PSNR={best['val_psnr_mean']:.4f} dB")
else:
    print("Pilot CSV not found. Run the B2 pilot first and publish its artifacts as denoise-B-results.")
print("Pilot train logs:", PILOT_LOGS if PILOT_LOGS else "none found")
print("Current shared LAMBDA_FROZEN:", PR.LAMBDA_FROZEN)
if RUN_STAGE == "train_G":
    assert PR.LAMBDA_FROZEN is not None, "Agree on lambda with Person A, commit/push common/protocol.py, and update PINNED_COMMIT first."
    assert PILOT_CSV.is_file(), "The B validation-only pilot must be available before G training."

In [ ]:
assert RUN_STAGE in {"review", "train_G", "infer", "verify"}, f"Unknown RUN_STAGE: {RUN_STAGE}"

if RUN_STAGE == "train_G":
    for seed in (0, 1):
        budget = min(MAX_HOURS_PER_RUN, SESSION_LIMIT_H - (time.time() - SESSION_START) / 3600 - 0.25)
        assert budget >= 0.5, "Insufficient session time to start the next G run."
        run([sys.executable, "-m", "scripts.train", "--arch", "unet", "--variant", "G",
             "--seed", str(seed), "--max-hours", f"{budget:.2f}"])
elif RUN_STAGE == "infer":
    for model_id in MODEL_IDS_TO_INFER:
        checkpoint = ARTIFACTS / "checkpoints" / model_id
        assert (checkpoint / "best.pt").is_file() and (checkpoint / "meta.json").is_file(), (
            f"Missing completed checkpoint for {model_id}; finish B2/B3 training and attach the results version.")
    for sigma in (15, 25, 50):
        noisy_dir = ARTIFACTS / "noisy" / f"sigma{sigma}"
        assert noisy_dir.is_dir() and len(list(noisy_dir.glob("*.png"))) >= 60, (
            f"Missing noisy test images for sigma {sigma}; attach the current-format shared artifacts dataset.")
    for model_id in MODEL_IDS_TO_INFER:
        run([sys.executable, "-m", "scripts.infer", "--method", "unet", "--model-id", model_id, "--device", "auto"])
elif RUN_STAGE == "verify":
    for model_id in MODEL_IDS_TO_INFER:
        checkpoint = ARTIFACTS / "checkpoints" / model_id
        missing = [name for name in ("best.pt", "last.pt", "meta.json", "train_log.csv")
                   if not (checkpoint / name).is_file()]
        assert not missing, f"{model_id} is missing: {missing}"
        meta = json.loads((checkpoint / "meta.json").read_text())
        print(model_id, "steps:", meta.get("step_reached"), "/", meta.get("steps_planned"),
              "status:", meta.get("status"), "best:", meta.get("best"))
        assert meta.get("status") == "done" and meta.get("step_reached") == meta.get("steps_planned"), (
            f"{model_id} is incomplete; resume the same training preset in B2/B3.")
        for sigma in (15, 25, 50):
            outputs = ARTIFACTS / "outputs" / model_id / f"sigma{sigma}"
            assert len(list(outputs.glob("*.png"))) == 60, f"Expected 60 {sigma} outputs for {model_id}."
        runtime_csv = ARTIFACTS / "results" / f"runtime_{model_id}.csv"
        assert runtime_csv.is_file(), f"Missing runtime file: {runtime_csv}"
        print("verified outputs and runtime:", model_id)

In [ ]:
provenance = []
for model_id in MODEL_IDS_TO_INFER:
    meta_path = ARTIFACTS / "checkpoints" / model_id / "meta.json"
    if not meta_path.is_file():
        continue
    meta = json.loads(meta_path.read_text())
    provenance.append({
        "model_id": model_id,
        "architecture": meta.get("arch"),
        "variant": meta.get("loss_variant"),
        "seed": meta.get("train_seed"),
        "params": meta.get("params"),
        "steps": meta.get("step_reached"),
        "lambda": meta.get("lambda"),
        "best_step": meta.get("best", {}).get("step"),
        "val_psnr": meta.get("best", {}).get("psnr_mean"),
        "hardware": meta.get("environment", {}).get("hardware"),
        "checkpoint_sha256": meta.get("checkpoint_sha256"),
    })
if provenance:
    provenance_df = pd.DataFrame(provenance)
    display(provenance_df)
    print("Use these verified meta.json values to update docs/model_provenance.md.")
else:
    print("No trained-model metadata found yet; finish training and import the denoise-B-results version.")
print("Publish these artifact folders as the next denoise-B-results version:")
for name in ("checkpoints", "outputs", "results", "logs"):
    folder = ARTIFACTS / name
    print(name, "present" if folder.is_dir() else "missing")